## M2 MobileNet OOD
The purpose of this notebook will be to try and change the MobileNet head to transfer learn it from ImageNet to CIFAR10 dataset. The aim is to implement a simple out-of-distribution (OOD) detection method using many different techniques and then test if it actually does something

<div style="text-align: center;">
<img src="https://www.researchgate.net/publication/376752661/figure/fig3/AS:11431281213995739@1703264754673/MobileNet-V1-architecture.png" width="900" alt="Mobile Net Architecture">
</div>

## 1. Setup
Import necessary libraries and set up the environment for training and evaluating the MobileNet model on the CIFAR10 dataset.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

### 1.1 Device Configuration
Set up the device for training (GPU if available, otherwise CPU).

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### 1.2 Getting training and testing dataset
Because MobileNet is pretrained on ImageNet, we need to resize the CIFAR10 images from 32x32 to 224x224 and normalize them using ImageNet's mean and std.

In [3]:
transform = transforms.Compose([
    transforms.Resize(224),  # MobileNet expects 224x224 images
    transforms.ToTensor(),  # Convert PIL image to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

train_dataset_id = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset_id = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader_id = DataLoader(train_dataset_id, batch_size=64, shuffle=True, num_workers=2)
test_loader_id = DataLoader(test_dataset_id, batch_size=64, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:08<00:00, 21.1MB/s] 


In [4]:
print(f"Number of training samples: {len(train_dataset_id)}")
print(f"Number of test samples: {len(test_dataset_id)}")

Number of training samples: 50000
Number of test samples: 10000


### 1.3 Getting the OOD dataset
For OOD detection, we will use the SVHN dataset and CIFAR100 as our out-of-distribution data

In [ ]:
ood_dataset_cifar100 = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
ood_loader_cifar100 = DataLoader(ood_dataset_cifar100, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR100 samples (OOD - CIFAR-100): {len(ood_dataset_cifar100)}")

In [ ]:
ood_dataset_svhn = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
ood_loader_svhn = DataLoader(ood_dataset_svhn, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of SVHN samples (OOD - SVHN): {len(ood_dataset_svhn)}")

### 1.4 Getting the model
For our test we will use MobileNet V3 Small pretrained on ImageNet

We will need to do the following:
- Load the pretrained MobileNet V3 Small model
- Freeze the feature extractor layers
- Replace the classifier head to output 10 classes for CIFAR-10

<div style="text-align: center;">
<img src="https://www.nim.org/fileadmin/PUBLIC/2_NIM_Forschung/mobilelogoshootout.png" width="900" alt="MobileNet vs Yolo">
</div>

In [ ]:
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
num_features = model.classifier[-1].in_features # Get the number of input features to the last layer
model.classifier[-1] = nn.Linear(num_features, 10) # Create a new linear layer with 10 outputs for CIFAR-10 and num_features inputs

In [ ]:
model = model.to(device)

## 2. Training the model
Set up the training loop to fine-tune the MobileNet model on the CIFAR-10 dataset.

### 2.1 Loss function and optimizer
Use Cross-Entropy Loss and Adam optimizer for training the model.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

### 2.2 Training loop

In [ ]:
epochs = 100

In [ ]:
for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    progress_bar = tqdm(train_loader_id, desc=f"Epochs {epoch+1}/{epochs}")

    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=running_loss / (progress_bar.n + 1))

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader_id):.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader_id:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy of the model on the 10000 test images: {accuracy:.2f}%")

## 3. OOD Detection
Implement simple OOD detection methods using the trained MobileNet model.

### 3.1 Energy-Based Scoring
Define a function to compute energy scores for OOD detection.
<div style="text-align: center;">
<img src="https://cherrypicked.dev/content/images/2020/10/Screen-Shot-2020-10-20-at-8.36.37-PM.png" width="900" alt="Energy Based OOD Detection">
</div>

In [ ]:
def get_energy_score(logits, T=1.0):
    """
    Compute the energy score for given logits.
    Lower energy indicates in-distribution.
    Higher energy indicates out-of-distribution.
    :param logits: Tensor of shape (N, C) where N is batch size and C is number of classes
    :param T: Temperature scaling parameter
    :return: Tensor of shape (N,) containing energy scores
    """
    energy = -T * torch.logsumexp(logits / T, dim=1)
    return energy

In [ ]:
def get_scores_for_loader(loader, model):
    model.eval()
    all_scores =
    with torch.no_grad():
        for inputs, _ in tqdm(loader, desc="Získávám OOD skóre"):
            inputs = inputs.to(device)
            logits = model(inputs)
            scores = get_energy_score(logits)
            all_scores.extend(scores.cpu().numpy())
    return np.array(all_scores)

In [ ]:
print("\nCalculating OOD scores for ID and OOD datasets...")
id_scores = get_scores_for_loader(test_loader_id, model)
ood_scores_cifar100 = get_scores_for_loader(ood_loader_cifar100, model)
ood_scores_svhn = get_scores_for_loader(ood_loader_svhn, model)

## 4.Plotting the results
Visualize the distribution of energy scores for in-distribution and out-of-distribution datasets.

In [ ]:
def plot_histograms(id_scores, ood_scores, id_label, ood_label, title):
    plt.figure(figsize=(10, 6))
    plt.hist(id_scores, bins=100, alpha=0.7, density=True, label=id_label)
    plt.hist(ood_scores, bins=100, alpha=0.7, density=True, label=ood_label)
    plt.xlabel("Energy Score (lower is better)")
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

plot_histograms(id_scores, ood_scores_cifar100,
                "ID (CIFAR-10)", "OOD (CIFAR-100)",
                "Energy Score Distribution: CIFAR-10 vs. CIFAR-100")

plot_histograms(id_scores, ood_scores_svhn,
                "ID (CIFAR-10)", "OOD (SVHN)",
                "Energy Score Distribution: CIFAR-10 vs. SVHN")